# P7513 assignment rate bar

This notebook recreates the visualization-stage assignment-rate plot for P7513 and extends it to compare original platform cell-boundary assignment against ProSeg assignment.

The original-mask rates use `table_xenium_cell_boundaries` and `table_merscope_cell_boundaries`, not nucleus masks. The ProSeg rates use `table_MOSAIK_proseg`.

## Kernel

Run this notebook with the MerXen environment that has `spatialdata` installed. On this machine, the workflow conda environment at `work/conda/env-5ab9b2fa7b005685690639bfde24c694` imports the plotting stack successfully.

In [ ]:
from pathlib import Path
import sys

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata as sd
from matplotlib.legend_handler import HandlerTuple
from matplotlib.patches import Patch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.plotting import save_figure

PLATFORM_COLORS = {
    "MERSCOPE": "#1f77b4",
    "XENIUM": "#d62728",
}
PLATFORM_ORDER = ("XENIUM", "MERSCOPE")


In [ ]:
PAIR_ID = "P7513"

MERSCOPE_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "merscope" / "latest" / "latest_spatialdata.zarr"
XENIUM_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "xenium" / "latest" / "latest_spatialdata.zarr"

OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "visualization" / "visualize_out"
OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_assignment_rate_bar_interactive.png"
OUTPUT_PDF_PATH = OUTPUT_PATH.with_suffix(".pdf")

print(f"MERSCOPE: {MERSCOPE_ZARR_PATH}")
print(f"Xenium:   {XENIUM_ZARR_PATH}")
print(f"PNG output: {OUTPUT_PATH}")
print(f"PDF output: {OUTPUT_PDF_PATH}")


In [ ]:
merscope_sdata = sd.read_zarr(MERSCOPE_ZARR_PATH)
xenium_sdata = sd.read_zarr(XENIUM_ZARR_PATH)

print("MERSCOPE tables:", list(merscope_sdata.tables.keys()))
print("Xenium tables:", list(xenium_sdata.tables.keys()))


In [ ]:
def _points_row_count(points_obj) -> int:
    """Return transcript count for pandas-like or dask-like points."""
    n_rows = points_obj.shape[0]
    if hasattr(n_rows, "compute"):
        n_rows = n_rows.compute()
    return int(n_rows)


def _table_count_sum(table_obj) -> int:
    """Return total assigned transcript counts represented by an AnnData table."""
    total = table_obj.X.sum()
    if hasattr(total, "compute"):
        total = total.compute()
    return int(np.asarray(total).item())


def _first_existing_table(sdata_obj, candidates: list[str]) -> str:
    for key in candidates:
        if key in sdata_obj.tables:
            return key
    raise KeyError(
        f"None of {candidates} found. Available tables: {list(sdata_obj.tables.keys())}"
    )


def assignment_rates_from_tables(sdata_obj, dataset_name: str) -> list[dict[str, object]]:
    """Compute original-mask and ProSeg assignment rates for one dataset."""
    dataset = dataset_name.upper()
    points_key = "transcripts" if "transcripts" in sdata_obj.points else list(sdata_obj.points.keys())[0]
    n_total = _points_row_count(sdata_obj.points[points_key])

    if dataset == "MERSCOPE":
        original_table = _first_existing_table(
            sdata_obj,
            ["table_merscope_cell_boundaries", "table_original"],
        )
    elif dataset == "XENIUM":
        original_table = _first_existing_table(
            sdata_obj,
            ["table_xenium_cell_boundaries", "table_original"],
        )
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    table_by_method = {
        "Original": original_table,
        "ProSeg": _first_existing_table(sdata_obj, ["table_MOSAIK_proseg", "table"]),
    }

    records = []
    for method, table_key in table_by_method.items():
        n_assigned = _table_count_sum(sdata_obj.tables[table_key])
        records.append(
            {
                "dataset": dataset,
                "method": method,
                "table_key": table_key,
                "n_transcripts_total": n_total,
                "n_transcripts_assigned": n_assigned,
                "pct_assigned": 100.0 * n_assigned / max(n_total, 1),
            }
        )
    return records


def build_assignment_rate_summary(merscope_sdata, xenium_sdata) -> pd.DataFrame:
    records = []
    records.extend(assignment_rates_from_tables(xenium_sdata, "XENIUM"))
    records.extend(assignment_rates_from_tables(merscope_sdata, "MERSCOPE"))
    return pd.DataFrame.from_records(records)


In [ ]:
assignment_df = build_assignment_rate_summary(merscope_sdata, xenium_sdata)
assignment_df


In [ ]:
from matplotlib.legend_handler import HandlerTuple


def _pale_color(color: str, *, white_fraction: float = 0.62) -> tuple[float, float, float]:
    """Blend a matplotlib color toward white for the original-mask bars."""
    rgb = np.array(mcolors.to_rgb(color), dtype=float)
    return tuple((1.0 - white_fraction) * rgb + white_fraction * np.ones(3))


def plot_assignment_bar_interactive(
    assignment_summaries: pd.DataFrame,
    output_path: Path | str | None = None,
    *,
    dataset_col: str = "dataset",
    method_col: str = "method",
    pct_col: str = "pct_assigned",
    platform_order: tuple[str, ...] = PLATFORM_ORDER,
    method_order: tuple[str, str] = ("Original", "ProSeg"),
    platform_colors: dict[str, str] = PLATFORM_COLORS,
    original_white_fraction: float = 0.62,
    bar_width: float = 0.36,
    group_spacing: float = 0.82,
    figsize: tuple[float, float] = (6.0, 4.2),
    value_fontsize: float = 9.0,
    xtick_fontsize: float = 10.0,
    ytick_fontsize: float = 10.0,
    xlabel_fontsize: float = 10.0,
    ylabel_fontsize: float = 10.0,
    legend_fontsize: float = 9.0,
) -> plt.Figure:
    """Plot paired original-mask and ProSeg assignment-rate bars."""
    required_cols = {dataset_col, method_col, pct_col}
    missing = required_cols.difference(assignment_summaries.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    present_datasets = list(dict.fromkeys(assignment_summaries[dataset_col].astype(str)))
    ordered_datasets = [name for name in platform_order if name in present_datasets]
    ordered_datasets.extend(name for name in present_datasets if name not in ordered_datasets)

    fig, ax = plt.subplots(figsize=figsize)
    x_positions = np.arange(len(ordered_datasets), dtype=float) * group_spacing
    method_offsets = {
        method_order[0]: -bar_width / 2.0,
        method_order[1]: bar_width / 2.0,
    }

    for dataset_index, dataset_name in enumerate(ordered_datasets):
        dataset_rows = assignment_summaries[
            assignment_summaries[dataset_col].astype(str) == dataset_name
        ]
        base_color = platform_colors.get(dataset_name.upper(), "#4b5563")
        for method in method_order:
            row = dataset_rows[dataset_rows[method_col].astype(str) == method]
            if row.empty:
                continue
            pct = float(row.iloc[0][pct_col])
            color = (
                _pale_color(base_color, white_fraction=original_white_fraction)
                if method == "Original"
                else base_color
            )
            x = x_positions[dataset_index] + method_offsets[method]
            ax.bar(
                x,
                pct,
                width=bar_width,
                color=color,
                edgecolor=base_color,
                linewidth=1.0,
            )
            ax.annotate(
                f"{pct:.1f}%",
                (x, pct),
                ha="center",
                va="bottom",
                fontsize=value_fontsize,
            )

    ax.set_ylim(0, 100)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [name.capitalize() for name in ordered_datasets],
        fontsize=xtick_fontsize,
    )
    ax.tick_params(axis="y", labelsize=ytick_fontsize)
    ax.set_ylabel("Assigned Transcripts (%)", fontsize=ylabel_fontsize)
    ax.set_xlabel("", fontsize=xlabel_fontsize)
    ax.set_title("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    legend_base_colors = [
        platform_colors.get(name.upper(), "#4b5563") for name in ordered_datasets
    ]
    legend_handles = [
        tuple(
            Patch(
                facecolor=_pale_color(color, white_fraction=original_white_fraction),
                edgecolor=color,
            )
            for color in legend_base_colors
        ),
        tuple(
            Patch(facecolor=color, edgecolor=color) for color in legend_base_colors
        ),
    ]
    ax.legend(
        handles=legend_handles,
        labels=["Original masks", "ProSeg"],
        handler_map={tuple: HandlerTuple(ndivide=None)},
        frameon=False,
        fontsize=legend_fontsize,
    )

    fig.tight_layout()
    if output_path is not None:
        save_figure(fig, output_path, dpi=220)
    return fig


In [ ]:
fig = plot_assignment_bar_interactive(
    assignment_df,
    output_path=None,
    original_white_fraction=0.62,
    bar_width=0.36,
    group_spacing=0.82,
    figsize=(6.0, 5),
    value_fontsize=15,
    xtick_fontsize=20,
    ytick_fontsize=15,
    xlabel_fontsize=20,
    ylabel_fontsize=20,
    legend_fontsize=15,
)


In [ ]:
# Optional: save the interactive output.
# save_figure() is the repo helper used by production plots; it writes PNG plus a sibling PDF.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_assignment_bar_interactive(
    assignment_df,
    output_path=OUTPUT_PATH,
    original_white_fraction=0.62,
    bar_width=0.36,
    group_spacing=0.82,
    figsize=(6.0, 5),
    value_fontsize=15,
    xtick_fontsize=20,
    ytick_fontsize=15,
    xlabel_fontsize=20,
    ylabel_fontsize=20,
    legend_fontsize=15,
)
print(f"Saved PNG: {OUTPUT_PATH}")
print(f"Saved PDF: {OUTPUT_PDF_PATH}")


In [ ]:
merscope_sdata.tables["table_original"]

In [ ]:
merscope_sdata.points["transcripts"]

In [ ]:
merscope_sdata.tables